<a href="https://colab.research.google.com/github/ajaykumar080286/DeepLearning/blob/master/44_integer_encoding_simplernn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
import numpy as np

import tensorflow

from tensorflow.keras.preprocessing.text import Tokenizer
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense,Embedding,Flatten
from keras.datasets import imdb
from keras.utils import pad_sequences

**Integer Encoding**

In [7]:
docs = ['go india',
		'india india',
		'hip hip hurray',
		'jeetega bhai jeetega india jeetega',
		'bharat mata ki jai',
		'kohli kohli',
		'sachin sachin',
		'dhoni dhoni',
		'modi ji ki jai',
		'inquilab zindabad']

In [8]:
docs

['go india',
 'india india',
 'hip hip hurray',
 'jeetega bhai jeetega india jeetega',
 'bharat mata ki jai',
 'kohli kohli',
 'sachin sachin',
 'dhoni dhoni',
 'modi ji ki jai',
 'inquilab zindabad']

In [9]:
tokenizer=Tokenizer(oov_token="<nothing>")

In [10]:
tokenizer.fit_on_texts(docs)

In [11]:
tokenizer.word_index

{'<nothing>': 1,
 'india': 2,
 'jeetega': 3,
 'hip': 4,
 'ki': 5,
 'jai': 6,
 'kohli': 7,
 'sachin': 8,
 'dhoni': 9,
 'go': 10,
 'hurray': 11,
 'bhai': 12,
 'bharat': 13,
 'mata': 14,
 'modi': 15,
 'ji': 16,
 'inquilab': 17,
 'zindabad': 18}

In [12]:
tokenizer.word_counts

OrderedDict([('go', 1),
             ('india', 4),
             ('hip', 2),
             ('hurray', 1),
             ('jeetega', 3),
             ('bhai', 1),
             ('bharat', 1),
             ('mata', 1),
             ('ki', 2),
             ('jai', 2),
             ('kohli', 2),
             ('sachin', 2),
             ('dhoni', 2),
             ('modi', 1),
             ('ji', 1),
             ('inquilab', 1),
             ('zindabad', 1)])

In [13]:
tokenizer.document_count

10

In [17]:
sequence=tokenizer.texts_to_sequences(docs)
sequence

[[10, 2],
 [2, 2],
 [4, 4, 11],
 [3, 12, 3, 2, 3],
 [13, 14, 5, 6],
 [7, 7],
 [8, 8],
 [9, 9],
 [15, 16, 5, 6],
 [17, 18]]

In [18]:
sequences=pad_sequences(sequence, padding="post")

In [19]:
sequences

array([[10,  2,  0,  0,  0],
       [ 2,  2,  0,  0,  0],
       [ 4,  4, 11,  0,  0],
       [ 3, 12,  3,  2,  3],
       [13, 14,  5,  6,  0],
       [ 7,  7,  0,  0,  0],
       [ 8,  8,  0,  0,  0],
       [ 9,  9,  0,  0,  0],
       [15, 16,  5,  6,  0],
       [17, 18,  0,  0,  0]], dtype=int32)

In [24]:
sequences[9] # we can get complte sentences

array([17, 18,  0,  0,  0], dtype=int32)

**Using Interger Encode Real Expample**

In [25]:
(X_train, y_train), (X_test, y_test)=imdb.load_data()

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [26]:
X_train.shape

(25000,)

In [31]:
len(X_train[3])

550

In [36]:
X_train[3]

array([  132,     8,    67,     6,    22,    15,     9,   283,     8,
        5168,    14,    31,     9,   242,   955,    48,    25,   279,
       22148,    23,    12,  1685,   195,    25,   238,    60,   796,
       13713,     4,   671,     7,  2804,     5,     4,   559,   154,
         888,     7,   726,    50,    26,    49,  7008,    15,   566,
          30,   579,    21,    64,  2574], dtype=int32)

In [33]:
X_train=pad_sequences(X_train, padding="post", maxlen=50) # we can take all full senences  then remove maxlen ,this is bcz training is fast in local system
X_test=pad_sequences(X_test, padding="post", maxlen=50)

In [35]:
X_test.shape

(25000, 50)

In [37]:
model=Sequential()
model.add(SimpleRNN(32, input_shape=(50,1),return_sequences=False))
model.add(Dense(1, activation="sigmoid"))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 32)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,121 (4.38 KB)

 Trainable params: 1,121 (4.38 KB)

 Non-trainable params: 0 (0.00 B)

In [40]:
model.compile(optimizer="adam", loss="binary_crossentropy",metrics=['accuracy'])

In [41]:
model.fit(X_train, y_train,epochs=5, validation_data=(X_test, y_test))

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step - accuracy: 0.5075 - loss: 0.6930 - val_accuracy: 0.5056 - val_loss: 0.6933
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.5098 - loss: 0.6929 - val_accuracy: 0.5061 - val_loss: 0.6944
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.5094 - loss: 0.6928 - val_accuracy: 0.5068 - val_loss: 0.6937
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - accuracy: 0.5068 - loss: 0.6929 - val_accuracy: 0.5030 - val_loss: 0.6938
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.5063 - loss: 0.6928 - val_accuracy: 0.5060 - val_loss: 0.6947
